In [ ]:
import requests
import json
from binascii import hexlify

def encrypt(texto_plano):
    texto_hex = hexlify(texto_plano).decode()
    url = "http://aes.cryptohack.org/ecb_oracle/encrypt/" + texto_hex + "/"
    respuesta = requests.get(url)
    return json.loads(respuesta.text)['ciphertext']

prueba = encrypt(b'AAAAAAAAAAAAAAAA')
print("Respuesta del servidor:", prueba)

Respuesta del servidor: ace317b30de0eb6d66cdbf8d50b1e1eaf67f5f7fad4f00797f4e2e678d61893f6f791fa103157b44ccfea4c321ad93f9


In [ ]:
import requests
import json
from binascii import hexlify
import time

def encrypt(texto_plano):
    texto_hex = hexlify(texto_plano).decode()
    url = "http://aes.cryptohack.org/ecb_oracle/encrypt/" + texto_hex + "/"
    for intento in range(10):
        try:
            respuesta = requests.get(url, timeout=15)
            datos = json.loads(respuesta.text)
            return datos['ciphertext']
        except Exception as e:
            print(f"Reintentando... ({intento+1}/10)")
            time.sleep(3)
    return None

print("Iniciando ataque ECB byte-at-a-time...")
print("=" * 50)

flag = ""
caracteres_posibles = "abcdefghijklmnopqrstuvwxyz0123456789_{}"

for i in range(40):
    pad_len = 15 - (i % 16)

    if pad_len == 0:
        pad_len = 16
        bloque_num = (i // 16) + 1
    else:
        bloque_num = i // 16

    padding = b'A' * pad_len
    inicio = bloque_num * 32
    fin = inicio + 32

    referencia = encrypt(padding)
    if referencia is None:
        print("Error de conexión.")
        break

    bloque_ref = referencia[inicio:fin]

    encontrado = False
    for caracter in caracteres_posibles:
        intento = encrypt(padding + flag.encode() + caracter.encode())
        if intento is None:
            continue

        bloque_intento = intento[inicio:fin]

        if bloque_ref == bloque_intento:
            flag += caracter
            print(f"Caracter encontrado: '{caracter}' → Flag: {flag}")
            encontrado = True
            break

        time.sleep(0.1)

    if not encontrado:
        print(f"No se encontró caracter en posición {i}")
        break
    if flag.endswith('}'):
        break

    time.sleep(0.8)

print("=" * 50)
print(f"FLAG FINAL: {flag}")


Iniciando ataque ECB byte-at-a-time...
Caracter encontrado: 'c' → Flag: c
Caracter encontrado: 'r' → Flag: cr
Caracter encontrado: 'y' → Flag: cry
Caracter encontrado: 'p' → Flag: cryp
Caracter encontrado: 't' → Flag: crypt
Caracter encontrado: 'o' → Flag: crypto
Caracter encontrado: '{' → Flag: crypto{
Caracter encontrado: 'p' → Flag: crypto{p
Caracter encontrado: '3' → Flag: crypto{p3
Caracter encontrado: 'n' → Flag: crypto{p3n
Caracter encontrado: '6' → Flag: crypto{p3n6
Caracter encontrado: 'u' → Flag: crypto{p3n6u
Caracter encontrado: '1' → Flag: crypto{p3n6u1
Caracter encontrado: 'n' → Flag: crypto{p3n6u1n
Caracter encontrado: '5' → Flag: crypto{p3n6u1n5
Caracter encontrado: '_' → Flag: crypto{p3n6u1n5_
Caracter encontrado: 'h' → Flag: crypto{p3n6u1n5_h
Caracter encontrado: '4' → Flag: crypto{p3n6u1n5_h4
Caracter encontrado: '7' → Flag: crypto{p3n6u1n5_h47
Caracter encontrado: '3' → Flag: crypto{p3n6u1n5_h473
Caracter encontrado: '_' → Flag: crypto{p3n6u1n5_h473_
Caracter encontr